# LLMScholar-Personas — Metrics Pipeline

Implements the exact metrics from LLMScholarBench (Espín-Noboa & Méndez, 2026):
- **Diversity** (normalized Shannon entropy, Eq. 9)
- **Parity** (1 − TV distance, Eq. 11)
- **Factuality** (matched / unique names, Eq. 5)
- **Consistency** (pairwise Jaccard across runs, Eq. 4)
- **Duplicates** (1 − unique/total, Eq. 3)

**Known gaps (flagged):**
- `created_at` is NaN in factuality_full → using `run_id` for ordering
- Author language not available → `diversity_language` skipped
- Gender and geography only available for *found* authors

## Step 0 — Setup and data loading

In [ ]:
from pathlib import Path
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_theme(style='whitegrid', context='notebook')

RESULTS      = Path('../../results')
FACT_PATH    = RESULTS / 'summary/factuality_full.csv'
ETH_GT_GLOB  = str(RESULTS / 'ethnicity/DataFrameRankings_Genderize_Namsor_*_with_ethnicity.csv')
SS_GT_PATH   = Path('/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet')
FIG_DIR      = RESULTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Call identity columns (one unique API call per combination + run_id)
CALL_KEYS = ['model', 'role', 'task', 'location', 'k', 'target', 'field', 'subfield', 'language', 'run_id']
# Prompt configuration (without run_id) — used for Consistency grouping
PROMPT_KEYS = [c for c in CALL_KEYS if c != 'run_id']

VALID_FLAGS = {'unchanged', 'cleaned', 'fixed_dict'}

In [ ]:
NEEDED_COLS = (
    # call identity
    ['model', 'role', 'task', 'location', 'k', 'target',
     'field', 'subfield', 'language', 'run_id']
    # validation / matching
    + ['valid_flag', 'name', 'lastname', 'author_status',
       'perceived_ethnicity', 'gt_gender', 'oa_country_code']
)

print('Loading factuality_full.csv (selected columns only) ...')
df = pd.read_csv(FACT_PATH, usecols=NEEDED_COLS, low_memory=False)
print(f'  Rows: {len(df):,}   Columns: {len(df.columns)}')
print(f'  valid_flag distribution:')
print(df['valid_flag'].value_counts().to_string())

In [ ]:
# ── Ground truth: ethnicity from CSVs (3 cols only), gender from parquet ──────
print('Loading ground-truth files ...')
gt_files = sorted(glob.glob(ETH_GT_GLOB))
if not gt_files:
    raise FileNotFoundError(f'No ground-truth files matched:\n  {ETH_GT_GLOB}')

# Load only the 3 columns we need — avoids pulling 15 GB into RAM
gt_parts = [
    pd.read_csv(f, usecols=['Researcher_id', 'Year', 'perceived_ethnicity'])
    for f in gt_files
]
gt = pd.concat(gt_parts, ignore_index=True)
del gt_parts  # free memory immediately

# Keep last year per researcher (deduplicated)
gt = gt.sort_values('Year').groupby('Researcher_id').last().reset_index()

# Gender from the deduplicated parquet (much smaller than re-reading all CSVs)
gt_parquet = pd.read_parquet(SS_GT_PATH, columns=['Researcher_id', 'Combined_gender'])
gt = gt.merge(gt_parquet, on='Researcher_id', how='left')
del gt_parquet

# Normalize gender
gt['gender_clean'] = gt['Combined_gender'].str.strip().str.lower().map(
    {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
)

# Normalize ethnicity
ETHNICITY_MAP = {
    'White': 'White',
    'Asian': 'Asian',
    'Black or African American': 'Black',
    'Hispanic or Latino': 'Hispanic',
    'American Indian or Alaska Native': 'American Indian',
}
gt['ethnicity_clean'] = gt['perceived_ethnicity'].map(ETHNICITY_MAP)

print(f'GT files loaded: {len(gt_files)}')
print(f'GT researchers (deduplicated): {len(gt):,}')
print('\nGT ethnicity distribution (excl. Unknown):')
gt_eth_dist = (gt['ethnicity_clean'].value_counts(normalize=True)
                 .rename('fraction').rename_axis('ethnicity'))
print(gt_eth_dist.to_string())
print('\nGT gender distribution (excl. Unknown):')
gt_gen_dist = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .rename('fraction').rename_axis('gender'))
print(gt_gen_dist.to_string())

In [ ]:
# ── Filter to valid responses only ───────────────────────────────────────────
valid = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'Valid rows: {len(valid):,} / {len(df):,}  ({len(valid)/len(df)*100:.1f}%)')

# Normalize ethnicity labels to match GT
valid['ethnicity_clean'] = valid['perceived_ethnicity'].map(ETHNICITY_MAP)

# Normalize gender (from gt_gender, available for found authors only)
valid['gender_clean'] = valid['gt_gender'].str.strip().str.lower().map(
    {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
)

# Author identifier per call
valid['author_id'] = valid['name'].fillna('') + ' ' + valid['lastname'].fillna('')
valid['author_id'] = valid['author_id'].str.strip()

print('\nMissing ethnicity after mapping:', valid['ethnicity_clean'].isna().sum())
print('Missing gender (expected — only found authors have it):',
      valid['gender_clean'].isna().sum(), '/',  len(valid))
print('Missing oa_country_code (expected — only found authors):',
      valid['oa_country_code'].isna().sum(), '/', len(valid))

## Model metadata (size, access, family)

In [ ]:
import re

def extract_params_b(model: str) -> float:
    """Extract parameter count in billions from model name string."""
    # MoE: 8x7b → 56B total, 8x22b → 176B
    moe = re.search(r'(\d+)x(\d+)b', model, re.I)
    if moe:
        return int(moe.group(1)) * int(moe.group(2))
    # Standard: 7b, 27b, 70b, 1.7b, 3.8b
    m = re.search(r'([\d.]+)b', model, re.I)
    if m:
        return float(m.group(1))
    return np.nan

def model_size_label(params_b: float) -> str:
    if np.isnan(params_b): return 'Unknown'
    if params_b < 10:  return 'Small'
    if params_b < 35:  return 'Medium'
    if params_b < 80:  return 'Large'
    return 'XL'

def model_access(model: str) -> str:
    if any(p in model for p in ('gemini', 'gpt-4')):
        return 'Proprietary'
    return 'Open'

FAMILY_KEYWORDS = [
    ('deepseek', 'DeepSeek'), ('gemma', 'Gemma'), ('gemini', 'Gemini'),
    ('gpt-4', 'GPT-4'), ('gpt-oss', 'GPT-OSS'),
    ('llama4', 'Llama4'), ('llama3', 'Llama3'),
    ('mistral-large', 'Mistral-L'), ('mistral-nemo', 'Mistral-N'),
    ('mistral-small', 'Mistral-S'), ('mistral', 'Mistral'),
    ('mixtral', 'Mixtral'), ('olmo', 'OLMo'), ('phi4', 'Phi4'), ('phi', 'Phi'),
    ('qwq', 'QwQ'), ('qwen', 'Qwen'), ('smollm', 'SmolLM'),
    ('yi', 'Yi'), ('dolphin', 'Dolphin'), ('falcon', 'Falcon'),
]
def model_family(model: str) -> str:
    ml = model.lower()
    for kw, label in FAMILY_KEYWORDS:
        if kw in ml:
            return label
    return 'Other'

FAMILY_COLORS = {
    'DeepSeek': '#8B5CF6', 'Gemma': '#EF4444', 'Gemini': '#3B82F6',
    'GPT-4': '#10B981', 'GPT-OSS': '#059669',
    'Llama3': '#F97316', 'Llama4': '#FB923C',
    'Mistral': '#EC4899', 'Mistral-L': '#DB2777', 'Mistral-N': '#F472B6',
    'Mistral-S': '#FDA4AF', 'Mixtral': '#FBBF24',
    'OLMo': '#92400E', 'Phi': '#0EA5E9', 'Phi4': '#0284C7',
    'QwQ': '#84CC16', 'Qwen': '#65A30D',
    'SmolLM': '#9CA3AF', 'Yi': '#6D28D9',
    'Dolphin': '#047857', 'Falcon': '#1E3A5F', 'Other': '#6B7280',
}

models_unique = valid['model'].dropna().unique()
model_meta = pd.DataFrame({'model': models_unique})
model_meta['params_b'] = model_meta['model'].map(extract_params_b)
model_meta['model_size'] = model_meta['params_b'].map(model_size_label)
model_meta['model_access'] = model_meta['model'].map(model_access)
model_meta['model_family'] = model_meta['model'].map(model_family)

print('Model metadata:')
display(model_meta.sort_values('params_b').reset_index(drop=True))

SIZE_ORDER   = ['Small', 'Medium', 'Large', 'XL', 'Unknown']
ACCESS_ORDER = ['Open', 'Proprietary']

valid = valid.merge(model_meta, on='model', how='left')

## Step 1 — Per-call metric functions

In [ ]:
ETH_CATS = ['Asian', 'Black', 'White', 'Hispanic', 'American Indian']
GEN_CATS = ['Female', 'Male', 'Neutral']


def normalized_shannon(counts: pd.Series) -> float:
    """Normalized Shannon entropy (Eq. 9). Excludes zeros."""
    n_cats = len(counts)
    if n_cats < 2:
        return np.nan
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(n_cats))


def total_variation(p_rec: pd.Series, q_gt: pd.Series) -> float:
    """Total Variation distance (Eq. 10): (1/2) * sum |p - q|."""
    cats = p_rec.index.union(q_gt.index)
    p = p_rec.reindex(cats, fill_value=0)
    q = q_gt.reindex(cats, fill_value=0)
    return float(0.5 * (p - q).abs().sum())


def compute_metrics_per_call(
    call_df: pd.DataFrame,
    gt_eth: pd.Series,
    gt_gen: pd.Series,
) -> dict:
    """
    Compute all metrics for a single API call (one group from CALL_KEYS).

    Parameters
    ----------
    call_df : rows belonging to one call (already filtered to valid responses)
    gt_eth  : ground-truth ethnicity fractions (Series indexed by ETH_CATS)
    gt_gen  : ground-truth gender fractions (Series indexed by GEN_CATS)

    Returns
    -------
    dict with keys: factuality, duplicates,
                    div_ethnicity, div_gender, div_geography,
                    parity_ethnicity, parity_gender, parity_geography
    """
    names_all    = call_df['author_id'].tolist()
    names_unique = call_df['author_id'].drop_duplicates().tolist()
    n_total  = len(names_all)
    n_unique = len(names_unique)

    if n_total == 0:
        return {}

    # ── Factuality (Eq. 5) ───────────────────────────────────────────────────
    n_found = (call_df.drop_duplicates('author_id')['author_status'] == 'found').sum()
    factuality = n_found / n_unique if n_unique > 0 else np.nan

    # ── Duplicates (Eq. 3) ───────────────────────────────────────────────────
    duplicates = 1 - (n_unique / n_total) if n_total > 0 else np.nan

    # ── Ethnicity diversity & parity ─────────────────────────────────────────
    eth_known = call_df['ethnicity_clean'].dropna()
    eth_counts = eth_known.value_counts().reindex(ETH_CATS, fill_value=0)
    eth_frac   = eth_counts / eth_counts.sum() if eth_counts.sum() > 0 else eth_counts.astype(float)
    div_eth    = normalized_shannon(eth_counts) if eth_counts.sum() >= 2 else np.nan
    tv_eth     = total_variation(eth_frac, gt_eth)
    parity_eth = 1 - tv_eth

    # ── Gender diversity & parity ─────────────────────────────────────────────
    gen_known  = call_df['gender_clean'].dropna()
    gen_counts = gen_known.value_counts().reindex(GEN_CATS, fill_value=0)
    gen_frac   = gen_counts / gen_counts.sum() if gen_counts.sum() > 0 else gen_counts.astype(float)
    div_gen    = normalized_shannon(gen_counts) if gen_counts.sum() >= 2 else np.nan
    tv_gen     = total_variation(gen_frac, gt_gen)
    parity_gen = 1 - tv_gen

    # ── Geography diversity & parity (OA country, found authors only) ─────────
    geo_known  = call_df['oa_country_code'].dropna()
    geo_unique_cats = geo_known.nunique()
    geo_counts = geo_known.value_counts()
    div_geo    = normalized_shannon(geo_counts) if geo_unique_cats >= 2 else np.nan
    # No GT geography distribution defined — parity_geography = NaN
    parity_geo = np.nan

    return {
        'n_total':          n_total,
        'n_unique':         n_unique,
        'n_found':          int(n_found),
        'factuality':       factuality,
        'duplicates':       duplicates,
        'div_ethnicity':    div_eth,
        'div_gender':       div_gen,
        'div_geography':    div_geo,
        'parity_ethnicity': parity_eth,
        'parity_gender':    parity_gen,
        'parity_geography': parity_geo,
    }

In [ ]:
# ── Build per-call metrics table ──────────────────────────────────────────────
# GT distributions (field-agnostic for now; could be filtered per field)
gt_eth_frac = (gt['ethnicity_clean'].dropna().value_counts(normalize=True)
                 .reindex(ETH_CATS, fill_value=0))
gt_gen_frac = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .reindex(GEN_CATS, fill_value=0))

print('Computing per-call metrics ...')
call_rows = []
groups = valid.groupby(CALL_KEYS, dropna=False)
print(f'Total calls: {groups.ngroups:,}')

for keys, g in groups:
    row = dict(zip(CALL_KEYS, keys))
    row.update(compute_metrics_per_call(g, gt_eth_frac, gt_gen_frac))
    call_rows.append(row)

calls = pd.DataFrame(call_rows)
calls = calls.merge(model_meta, on='model', how='left')

print(f'Calls table: {len(calls):,} rows × {len(calls.columns)} columns')
print('\nSample:')
metric_cols = ['factuality','duplicates','div_ethnicity','div_gender',
               'div_geography','parity_ethnicity','parity_gender']
display(calls[['model','language','location','field','run_id'] + metric_cols].head(10))

## Step 2 — Consistency (Eq. 4)

In [ ]:
def compute_consistency(
    responses_df: pd.DataFrame,
    group_by: list = PROMPT_KEYS,
) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity between consecutive runs of the same prompt (Eq. 4).

    Groups by `group_by`, sorts by run_id, computes Jaccard for each
    consecutive pair (i, i-1), returns mean per group.
    """
    rows = []
    for keys, g in responses_df.groupby(group_by, dropna=False):
        g_sorted = g.sort_values('run_id')
        run_sets = [
            set(rg['author_id'].dropna())
            for _, rg in g_sorted.groupby('run_id', sort=True)
        ]
        if len(run_sets) < 2:
            continue
        jaccards = []
        for i in range(1, len(run_sets)):
            a, b = run_sets[i - 1], run_sets[i]
            union = a | b
            jaccards.append(len(a & b) / len(union) if union else np.nan)
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n_runs'] = len(run_sets)
        row['consistency'] = float(np.nanmean(jaccards)) if jaccards else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


print('Computing consistency ...')
cons_df = compute_consistency(valid)
cons_df = cons_df.merge(model_meta, on='model', how='left')
print(f'Consistency rows: {len(cons_df):,}')
print(f'Mean consistency: {cons_df["consistency"].mean():.4f}')
display(cons_df.groupby('model')['consistency'].agg(['mean','median','count']).round(4))

In [ ]:
# ── Merge consistency into calls table ───────────────────────────────────────
calls = calls.merge(
    cons_df[PROMPT_KEYS + ['consistency']],
    on=PROMPT_KEYS, how='left'
)
print('Calls with consistency:', calls['consistency'].notna().sum(), '/', len(calls))

## Step 3 — Aggregation

In [ ]:
METRIC_COLS = ['factuality', 'duplicates', 'consistency',
               'div_ethnicity', 'div_gender', 'div_geography',
               'parity_ethnicity', 'parity_gender']


def ci95(series: pd.Series) -> float:
    """95% CI half-width using Student t-distribution."""
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    return float(stats.t.ppf(0.975, df=len(s) - 1) * s.sem())


def aggregate_scores(
    all_calls_df: pd.DataFrame,
    group_by: list,
    metrics: list = METRIC_COLS,
) -> pd.DataFrame:
    """
    Flexible aggregation with mean ± 95% CI.

    Parameters
    ----------
    all_calls_df : per-call metrics table
    group_by     : list of columns to group by (e.g. ['model_size', 'language'])
    metrics      : metric columns to aggregate

    Returns
    -------
    DataFrame with columns: group_by cols | metric_mean | metric_ci | n
    """
    rows = []
    for keys, g in all_calls_df.groupby(group_by, dropna=False):
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n'] = len(g)
        for m in metrics:
            s = g[m].dropna()
            row[f'{m}_mean'] = s.mean() if len(s) else np.nan
            row[f'{m}_ci']   = ci95(s)
        rows.append(row)
    return pd.DataFrame(rows)


# ── Key aggregations ─────────────────────────────────────────────────────────
agg_model      = aggregate_scores(calls, ['model', 'model_size', 'model_access', 'model_family'])
agg_size       = aggregate_scores(calls, ['model_size'])
agg_access     = aggregate_scores(calls, ['model_access'])
agg_size_lang  = aggregate_scores(calls, ['model_size', 'language'])
agg_size_loc   = aggregate_scores(calls, ['model_size', 'location'])

print('By model size:')
display(agg_size[['model_size'] + [c for c in agg_size.columns if c.endswith('_mean')]])

## Step 5 — Summary table

In [ ]:
summary = aggregate_scores(
    calls,
    group_by=['model', 'model_size', 'model_access', 'field', 'language', 'location'],
)
mean_cols = [c for c in summary.columns if c.endswith('_mean')]
summary_out = summary.rename(columns={c: c.replace('_mean', '') for c in mean_cols})

# Flag NaN combinations
n_empty = summary_out[METRIC_COLS].isna().all(axis=1).sum()
if n_empty:
    print(f'WARNING: {n_empty} model×task combinations with all-NaN metrics')

print(f'Summary table: {len(summary_out):,} rows')
display(summary_out.head(10))

## Step 4 — Plots

In [ ]:

def plot_grouped_metrics(
    all_calls_df,
    group_configs,
    metrics=None,
    metric_directions=None,
    figsize=None,
    save_path=None,
):
    """
    Horizontal bar chart replicating Figure 2 of LLMScholarBench.

    Each group_config = one visual section (rows = group values).
    Each metric = one column of bars (mean ± 95% CI, Student t).
    Best value per section×metric shown in bold (per direction).
    n < 3 samples: '*' suffix, no CI bar.
    Colors within a section go from light to dark following row order.
    """
    from scipy import stats
    import matplotlib.colors as mcolors

    DEFAULT_METRICS = [
        'refusals', 'validity', 'duplicates', 'consistency',
        'factuality', 'connectedness', 'similarity', 'diversity', 'parity',
    ]
    DEFAULT_DIRECTIONS = {
        'refusals': None, 'validity': '↑', 'duplicates': '↓',
        'consistency': None, 'factuality': '↑', 'connectedness': None,
        'similarity': None, 'diversity': None, 'parity': '↑',
        'div_gender': None, 'div_ethnicity': None,
        'div_language': None, 'div_geography': None,
        'parity_gender': '↑', 'parity_ethnicity': '↑',
        'parity_language': '↑', 'parity_geography': '↑',
    }

    if metrics is None:
        metrics = [m for m in DEFAULT_METRICS if m in all_calls_df.columns]
    dirs = {**DEFAULT_DIRECTIONS, **(metric_directions or {})}

    def _ci95(s):
        s = s.dropna()
        if len(s) < 2:
            return np.nan
        return float(stats.t.ppf(0.975, df=len(s) - 1) * s.sem())

    def _shades(hex_color, n):
        base = np.array(mcolors.to_rgb(hex_color))
        white = np.ones(3)
        if n == 1:
            return [tuple(white * 0.25 + base * 0.75)]
        return [tuple(white * (1 - t) + base * t) for t in np.linspace(0.35, 1.0, n)]

    # ── Build sections ────────────────────────────────────────────────────────
    sections = []
    for gc in group_configs:
        col = gc['column']
        if col not in all_calls_df.columns:
            print(f"Warning: column '{col}' not in dataframe — skipping section '{gc['label']}'.")
            continue
        available = set(all_calls_df[col].dropna().unique())
        if 'order' in gc:
            order = [v for v in gc['order'] if v in available]
            order += sorted([v for v in available if v not in gc['order']], key=str)
        else:
            order = sorted(available, key=str)

        rows = []
        for val in order:
            grp = all_calls_df[all_calls_df[col] == val]
            row = {'label': str(val)}
            for m in metrics:
                if m not in grp.columns:
                    row[f'{m}_mean'] = np.nan
                    row[f'{m}_ci']   = np.nan
                    row[f'{m}_n']    = 0
                else:
                    s = grp[m].dropna()
                    row[f'{m}_mean'] = float(s.mean()) if len(s) else np.nan
                    row[f'{m}_ci']   = _ci95(s)
                    row[f'{m}_n']    = len(s)
            rows.append(row)

        if rows:
            sections.append({'label': gc['label'], 'color': gc['color'], 'rows': rows})

    if not sections or not metrics:
        raise ValueError("No valid sections or metrics to plot.")

    # ── Y-position layout (top-to-bottom via invert_yaxis) ───────────────────
    SECTION_GAP = 0.8
    y = 0.0
    y_lookup   = {}   # (si, ri) -> y_val
    sec_ranges = []   # (y_top, y_bot) per section
    for si, sec in enumerate(sections):
        if si > 0:
            y += SECTION_GAP
        y_top = y
        for ri in range(len(sec['rows'])):
            y_lookup[(si, ri)] = y
            y += 1.0
        sec_ranges.append((y_top, y - 1.0))

    y_data_max = y - 1.0
    sep_ys = [
        (sec_ranges[i][1] + sec_ranges[i + 1][0]) / 2
        for i in range(len(sections) - 1)
    ]

    # ── Figsize ───────────────────────────────────────────────────────────────
    n_m = len(metrics)
    total_rows = sum(len(s['rows']) for s in sections)
    if figsize is None:
        w = 2.2 * n_m + 2.5
        h = max(0.5 * total_rows + 0.8 * len(sections) + 1.0, 3.0)
        figsize = (w, h)

    fig, axes = plt.subplots(
        1, n_m + 1, figsize=figsize,
        gridspec_kw={'width_ratios': [1.8] + [2.2] * n_m},
    )
    axes = np.atleast_1d(axes)

    pad  = 0.6
    y_lo = 0.0 - pad
    y_hi = y_data_max + pad

    # ── Left label panel ──────────────────────────────────────────────────────
    lax = axes[0]
    lax.set_xlim(0, 1)
    lax.set_ylim(y_lo, y_hi)
    lax.invert_yaxis()
    lax.axis('off')

    for si, (sec, (y_top, y_bot)) in enumerate(zip(sections, sec_ranges)):
        y_c = (y_top + y_bot) / 2
        lax.text(0.08, y_c, sec['label'], ha='center', va='center',
                 fontsize=8.5, fontweight='bold', multialignment='center')
        bx = 0.30
        lax.plot([bx, bx], [y_top - 0.3, y_bot + 0.3], color='#444', lw=1.1)
        lax.plot([bx, bx + 0.05], [y_top - 0.3, y_top - 0.3], color='#444', lw=1.1)
        lax.plot([bx, bx + 0.05], [y_bot + 0.3, y_bot + 0.3], color='#444', lw=1.1)
        for ri, row in enumerate(sec['rows']):
            lax.text(0.98, y_lookup[(si, ri)], row['label'],
                     ha='right', va='center', fontsize=7.5)

    # ── Metric axes ───────────────────────────────────────────────────────────
    for m, ax in zip(metrics, axes[1:]):
        direction = dirs.get(m)
        arrow = f' {direction}' if direction else ''
        nice  = (m.replace('_', ' ')
                  .replace('div ', 'Div ')
                  .replace('parity ', 'Par ')
                  .title())
        ax.set_title(f'{nice}{arrow}', fontsize=8, fontweight='bold', pad=4)
        ax.set_xlim(0, 1)
        ax.set_ylim(y_lo, y_hi)
        ax.invert_yaxis()
        ax.set_yticks([])
        ax.set_xticks([0, 0.5, 1])
        ax.tick_params(axis='x', labelsize=7)
        ax.spines[['left', 'right', 'top']].set_visible(False)
        ax.grid(axis='x', alpha=0.25, zorder=0)

        for sy in sep_ys:
            ax.axhline(sy, color='#bbb', lw=0.9, ls='--', zorder=1)

        for si, (sec, (y_top, y_bot)) in enumerate(zip(sections, sec_ranges)):
            n_rows = len(sec['rows'])
            colors = _shades(sec['color'], n_rows)
            vals = [row[f'{m}_mean'] for row in sec['rows']]
            cis  = [row[f'{m}_ci']   for row in sec['rows']]
            ns   = [row[f'{m}_n']    for row in sec['rows']]

            good = [(i, v) for i, v in enumerate(vals) if not np.isnan(v)]
            if direction == '↑' and good:
                best = max(v for _, v in good)
            elif direction == '↓' and good:
                best = min(v for _, v in good)
            else:
                best = None

            for ri, (val, ci, n, color) in enumerate(zip(vals, cis, ns, colors)):
                yv = y_lookup[(si, ri)]
                if np.isnan(val):
                    continue

                is_best = best is not None and abs(val - best) < 1e-9
                low_n   = n < 3

                ax.barh(yv, val, height=0.55, color=color, alpha=0.92, zorder=2)

                if not np.isnan(ci) and not low_n:
                    ax.errorbar(val, yv, xerr=ci, fmt='none',
                                color='#222', lw=0.9, capsize=2, zorder=3)

                txt = f'{val:.2f}{"*" if low_n else ""}'
                fw  = 'bold' if is_best else 'normal'
                if val > 0.18:
                    ax.text(val - (ci or 0) - 0.02, yv, txt,
                            ha='right', va='center', fontsize=7,
                            fontweight=fw, zorder=4)
                else:
                    ax.text(val + (ci or 0) + 0.02, yv, txt,
                            ha='left', va='center', fontsize=7,
                            fontweight=fw, zorder=4)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f'Saved → {save_path}')

    plt.show()
    return fig


print('plot_grouped_metrics ready.')


In [ ]:

# Plot 1 — Replicar Figure 2 del paper (infraestructura)
# Nota: 'reasoning' no está en calls todavía — se añade si hay datos de esa columna
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Access', 'column': 'model_access',
         'color': '#4A90D9', 'order': ['Open', 'Proprietary']},
        {'label': 'Size',   'column': 'model_size',
         'color': '#5DB85D', 'order': ['Small', 'Medium', 'Large', 'XL']},
    ],
    metrics=['factuality', 'duplicates', 'consistency',
             'div_gender', 'div_ethnicity', 'div_geography',
             'parity_gender', 'parity_ethnicity'],
    save_path=str(FIG_DIR / 'plot_infrastructure.png'),
)


In [ ]:

# Plot 2 — Solo Diversity, agrupado por Language y Location
# 'geography' no es una columna en calls; la columna del prompt es 'location'
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Language', 'column': 'language', 'color': '#E8A838'},
        {'label': 'Location', 'column': 'location', 'color': '#D95B5B'},
    ],
    metrics=['div_gender', 'div_ethnicity', 'div_geography'],
    save_path=str(FIG_DIR / 'plot_diversity_lang_geo.png'),
)


In [ ]:

# Plot 3 — Solo Parity, agrupado por Language y Location
# parity_geography es NaN para todos (sin GT de geografía) — se omite
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Language', 'column': 'language', 'color': '#E8A838'},
        {'label': 'Location', 'column': 'location', 'color': '#D95B5B'},
    ],
    metrics=['parity_gender', 'parity_ethnicity'],
    save_path=str(FIG_DIR / 'plot_parity_lang_geo.png'),
)


In [ ]:

# Plot 4 — Factuality y Duplicates por model size
# Las sub-métricas factuality_author/field/epoch/seniority no están en calls todavía
# (requieren join con factuality_metrics.ipynb); se usa factuality global por ahora
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Size', 'column': 'model_size',
         'color': '#5DB85D', 'order': ['Small', 'Medium', 'Large', 'XL']},
    ],
    metrics=['factuality', 'duplicates'],
    save_path=str(FIG_DIR / 'plot_factuality.png'),
)


In [ ]:

# Plot 5 — Consistency por Language y Location
plot_grouped_metrics(
    calls,
    group_configs=[
        {'label': 'Language', 'column': 'language', 'color': '#E8A838'},
        {'label': 'Location', 'column': 'location', 'color': '#D95B5B'},
    ],
    metrics=['consistency'],
    save_path=str(FIG_DIR / 'plot_consistency_lang_geo.png'),
)
